In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.chdir('drive/MyDrive/AI+X고급/식물을 부탁해/')

## 새로운 데이터셋 만들때 아래의 2개 셀 수정 필수

In [3]:
file_name = "TS_A.화초_1.건생식물_03.금전수"
plant_name = "금전수"

In [4]:
# 라벨링 데이터셋 불러오기
# (1) 드라이브 -> Colab으로 복사
!cp "/content/drive/MyDrive/AI+X고급/식물을 부탁해/labelling_data/TL_A.화초_1.건생식물_03.금전수.zip" "/content/"

# (2) Colab에서 압축 해제
!unzip -q "/content/TL_A.화초_1.건생식물_03.금전수.zip" -d "/content/TL_A.화초_1.건생식물_03.금전수"

# (3) 압축 파일 삭제 (용량 절약)
!rm "/content/TL_A.화초_1.건생식물_03.금전수.zip"

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.


In [5]:
# (1) 드라이브 → Colab으로 복사
!cp "/content/drive/MyDrive/AI+X고급/식물을 부탁해/TS_A.화초_1.건생식물_03.금전수.zip" "/content/"

# (2) Colab에서 압축 해제
!unzip -q "/content/TS_A.화초_1.건생식물_03.금전수.zip" -d "/content/TS_A.화초_1.건생식물_03.금전수"

# (3) 압축 파일 삭제 (용량 절약)
!rm "/content/TS_A.화초_1.건생식물_03.금전수.zip"

## 데이터셋 정보 설정

In [6]:
import os, zipfile, shutil, random, io
from pathlib import Path
import pandas as pd

# ====== 사용자 설정 ======
SRC_FOLDER = f"/content/{file_name}"           # 원본 폴더 경로
OUT_ZIP = Path(f"/content/{plant_name}_dataset.zip")  # 최종 결과 ZIP
RANDOM_SEED = 42
# ========================

CLASSES   = ["underwatered", "normal", "overwatered"]
IMG_EXTS  = ".jpg"

## 데이터셋 폴더 만들기

In [7]:
def get_true_label_from_filename(img_name: str) -> str:
    """
    파일명 패턴에서 GT 클래스 추출
      'B-1-?' 또는 'L-1-?' → underwatered
      'B-2-?' 또는 'L-2-?' → normal
      'B-3-?' 또는 'L-3-?' → overwatered
    """
    tokens = img_name.split("-")
    for i in range(1, len(tokens) - 1):
        if tokens[i] in {"1","2","3"} and tokens[i-1].isalpha() and tokens[i+1].isalpha():
            return {"1":"underwatered", "2":"normal", "3":"overwatered"}[tokens[i]]
    raise ValueError(f"GT 클래스를 추출할 수 없음: {img_name}")

def scan_folder_entries(src_folder: Path):
    """폴더 내부 이미지 항목을 스캔하여 (filepath, img_name, label) 리스트 반환"""
    items = []
    skipped_nonimg = 0
    skipped_label = 0

    for root, _, files in os.walk(src_folder):
        for fname in files:
            ext = Path(fname).suffix.lower()
            if ext != IMG_EXTS:
                skipped_nonimg += 1
                continue
            try:
                label = get_true_label_from_filename(fname)
            except Exception:
                skipped_label += 1
                continue
            items.append((os.path.join(root, fname), fname, label))

    return items, skipped_nonimg, skipped_label

def downsample_items(items, seed=42):
    """
    items: list of (filepath, img_name, label)
    모든 클래스를 최소 개수로 다운샘플링해서 균형 맞춤.
    """
    df = pd.DataFrame(items, columns=["filepath", "img_name", "label"])
    before_counts = df['label'].value_counts().sort_index()

    if before_counts.empty:
        raise RuntimeError("이미지 항목을 찾지 못했습니다. 파일 확장자 또는 파일명 패턴을 확인하세요.")

    min_count = before_counts.min()
    rng = random.Random(seed)

    balanced_rows = []
    for cls in CLASSES:
        cls_df = df[df['label'] == cls]
        if len(cls_df) == 0:
            continue
        idxs = cls_df.index.tolist()
        k = min(min_count, len(idxs))  # 클래스 개수가 적으면 가능한 만큼만 사용
        chosen = rng.sample(idxs, k=k)
        balanced_rows.append(cls_df.loc[chosen])

    if len(balanced_rows) == 0:
        raise RuntimeError("다운샘플링 대상이 없습니다.")

    df_balanced = pd.concat(balanced_rows).sample(frac=1, random_state=seed).reset_index(drop=True)
    after_counts = df_balanced['label'].value_counts().sort_index()
    return df, before_counts, df_balanced, after_counts

def write_balanced_items_to_zip(df_balanced: pd.DataFrame, out_zip_path: Path):
    """
    다운샘플링된 항목만 하나의 ZIP으로 기록.
    ZIP 내부 경로: train/<label>/<filename>
    """
    if out_zip_path.exists():
        out_zip_path.unlink()

    written_counts = {c: 0 for c in CLASSES}
    seen_names = set()  # ZIP 내부 파일명 중복 방지

    with zipfile.ZipFile(out_zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zout:
        for idx, row in df_balanced.iterrows():
            fpath  = row['filepath']
            fname  = row['img_name']
            label  = row['label']

            # ZIP 내부 경로 (중복 시 _1, _2 ... 접미사)
            arcname = f"train/{label}/{fname}"
            if arcname in seen_names:
                stem = Path(fname).stem
                suf  = Path(fname).suffix
                k = 1
                while arcname in seen_names:
                    arcname = f"train/{label}/{stem}_{k}{suf}"
                    k += 1
            seen_names.add(arcname)

            # 로컬 파일에서 읽어 결과 ZIP에 기록
            with open(fpath, 'rb') as fin:
                data = fin.read()
                zout.writestr(arcname, data)

            written_counts[label] += 1

            # 진행 로그(간단)
            if (idx + 1) % 1000 == 0:
                print(f"  - {idx+1}/{len(df_balanced)} 이미지 기록 중...")

    return written_counts

def main():
    print(f"[1/3] 폴더 스캔: {SRC_FOLDER}")
    items, skipped_nonimg, skipped_label = scan_folder_entries(SRC_FOLDER)
    print(f"  - 이미지 항목 수: {len(items)}")
    print(f"  - 스킵(비이미지): {skipped_nonimg}")
    print(f"  - 스킵(라벨 추출 실패): {skipped_label}")

    print("\n[2/3] 클래스 분포 및 다운샘플링")
    df_all, before_counts, df_balanced, after_counts = downsample_items(items, seed=RANDOM_SEED)
    print("  - 다운샘플링 전:")
    print(before_counts)
    print("\n  - 다운샘플링 후(균형):")
    print(after_counts)

    print(f"\n[3/3] ZIP으로 기록 → {OUT_ZIP}")
    written = write_balanced_items_to_zip(df_balanced, OUT_ZIP)
    for c in CLASSES:
        print(f"  - {c:13s}: {written.get(c,0)} files")

    print(f"\n완료 ✅ 결과 ZIP: {OUT_ZIP}")

if __name__ == "__main__":
    main()


[1/3] 폴더 스캔: /content/TS_A.화초_1.건생식물_03.금전수
  - 이미지 항목 수: 20000
  - 스킵(비이미지): 0
  - 스킵(라벨 추출 실패): 0

[2/3] 클래스 분포 및 다운샘플링
  - 다운샘플링 전:
label
normal          7070
overwatered     6251
underwatered    6679
Name: count, dtype: int64

  - 다운샘플링 후(균형):
label
normal          6251
overwatered     6251
underwatered    6251
Name: count, dtype: int64

[3/3] ZIP으로 기록 → /content/금전수_dataset.zip
  - 1000/18753 이미지 기록 중...
  - 2000/18753 이미지 기록 중...
  - 3000/18753 이미지 기록 중...
  - 4000/18753 이미지 기록 중...
  - 5000/18753 이미지 기록 중...
  - 6000/18753 이미지 기록 중...
  - 7000/18753 이미지 기록 중...
  - 8000/18753 이미지 기록 중...
  - 9000/18753 이미지 기록 중...
  - 10000/18753 이미지 기록 중...
  - 11000/18753 이미지 기록 중...
  - 12000/18753 이미지 기록 중...
  - 13000/18753 이미지 기록 중...
  - 14000/18753 이미지 기록 중...
  - 15000/18753 이미지 기록 중...
  - 16000/18753 이미지 기록 중...
  - 17000/18753 이미지 기록 중...
  - 18000/18753 이미지 기록 중...
  - underwatered : 6251 files
  - normal       : 6251 files
  - overwatered  : 6251 files

완료 ✅ 결과 ZIP: /content/금전수_da

In [11]:
zip_path = f"/content/{plant_name}_dataset"

dest_path = f"/content/drive/MyDrive/AI+X고급/식물을 부탁해/kjw/dataset(base)/"
!cp "{zip_path}.zip" "{dest_path}"

print("✅ 드라이브 복사 완료:", dest_path)

✅ 드라이브 복사 완료: /content/drive/MyDrive/AI+X고급/식물을 부탁해/kjw/dataset(base)/


In [12]:
# # 만약 용량이 부족하여 rm: cannot remove '/content/???': Is a directory라고 뜬 경우 실행 후 다시 바로 위의 셀 실행
# !rm -rf "/content/TS_A.화초_1.건생식물_03.금전수"